# Triadic Cell Notebook v60
## Cached KRRB Controller Search

v59 had the right concept but the wrong runtime shape.

The slow section repeatedly recomputed wrong-slot branches during the parameter sweep. v60 scores once, caches a branch tensor, and then sweeps controller logic cheaply.

Core correction:

$$
\boxed{\text{raw geometry drives; residual audits}}
$$

Target:

$$
\boxed{\text{hurt}=0 \land \text{accuracy}>\text{base}}
$$


In [ ]:
from __future__ import annotations

import os, json, random, re, math
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams["figure.figsize"]=(10,5)
plt.rcParams["axes.grid"]=True
np.set_printoptions(suppress=True, precision=4)

DATA_JSONL=""
MAX_SAMPLES=128

HF_TOKEN=os.getenv("HF_TOKEN","")
LOCAL_FILES_ONLY=True

MODEL_PROFILE="qwen25_1p5b_instruct"
MODEL_PROFILES={
    "qwen25_1p5b_instruct":{
        "MODEL_NAME":"Qwen/Qwen2.5-1.5B-Instruct",
        "EMBED_MODEL_NAME":"sentence-transformers/all-MiniLM-L6-v2",
        "MAX_SAMPLES_CAP":128,
    },
    "qwen25_3b_instruct":{
        "MODEL_NAME":"Qwen/Qwen2.5-3B-Instruct",
        "EMBED_MODEL_NAME":"sentence-transformers/all-MiniLM-L6-v2",
        "MAX_SAMPLES_CAP":96,
    },
}

profile=MODEL_PROFILES[MODEL_PROFILE]
MODEL_NAME=profile["MODEL_NAME"]
EMBED_MODEL_NAME=profile["EMBED_MODEL_NAME"]
MAX_SAMPLES=min(MAX_SAMPLES, profile["MAX_SAMPLES_CAP"])

USE_CHAT_TEMPLATE=True
CHOICE_AUDIT_MODE="rotations"

ALPHA_SWEEP=[0.0,0.25,0.5,0.75,1.0,1.25,1.5]
SUPPORT_MIN_SWEEP=[1,2,3]
MARGIN_MIN_SWEEP=[0.00,0.05,0.10,0.20]
BASE_LOCK_MARGIN=4.0

# Final chosen defaults after sweep inspection.
FINAL_ALPHA=0.75
FINAL_SUPPORT_MIN=2
FINAL_MARGIN_MIN=0.05

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32

OUTPUT_DIR=f"v59_outputs_{MODEL_PROFILE}_{CHOICE_AUDIT_MODE}_family_quotient"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")
print("MODEL:", MODEL_NAME)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# v60 overrides
OUTPUT_DIR=f"v60_outputs_{MODEL_PROFILE}_{CHOICE_AUDIT_MODE}_cached_controller_search"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

RAW_MARGIN_SWEEP=[0.00,0.05,0.10,0.20]
Q_MARGIN_SWEEP=[0.00,0.05,0.10,0.20]
DISAGREE_POLICY_SWEEP=["omega","raw","base"]

print("v60 OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
ADVERSARIAL_NEXUS_DATA = [{'id': 'adv_coupler_01', 'band': 'inverse_need_adversarial', 'prompt': 'A spinning rubber coupler is loose on a vacuum pump shaft. The repair must add radial compression while keeping the coupler centered enough to transmit rotation. Which candidate is operationally best?', 'choices': ['tight O-rings seated concentrically around the coupler', 'a poetic recursive wrap that symbolically surrounds the failure', 'loose string nearby because string can wrap objects', 'permanent epoxy locking the coupler off-center'], 'answer_idx': 0}, {'id': 'adv_coupler_02', 'band': 'inverse_need_adversarial', 'prompt': "The missing function is not the noun 'rubber part'; it is centered compressive coupling under motion. Which answer preserves that function with the least overbinding?", 'choices': ['a removable radial compression band', 'a same-named replacement label with no fit data', 'a clamp that crushes one side harder than the other', 'a larger motor housing'], 'answer_idx': 0}, {'id': 'adv_car_01', 'band': 'interface_adversarial', 'prompt': 'A car hides combustion, gearing, sensors, tire friction, steering geometry, and safety constraints. What is the correct interface-collapse?', 'choices': ['a semantic category called vehicle', 'a readable driver surface: wheel, pedals, seat, motion', 'a detailed list of engine nouns', 'a symbol of transportation culture'], 'answer_idx': 1}, {'id': 'adv_house_01', 'band': 'fold_adversarial', 'prompt': 'A house presents door, room, roof, and shelter. Which answer captures the hidden inward fold rather than the surface noun?', 'choices': ['a building label recognized by zoning language', 'weather, privacy, load, heat-flow, wiring, plumbing, and human paths folded into shelter', 'a decorative facade with rooms inside', 'a static object that stops being computational'], 'answer_idx': 1}, {'id': 'adv_api_01', 'band': 'interface_adversarial', 'prompt': 'An API call exposes one method while hiding authentication, routing, validation, persistence, retries, and errors. What is the operational event?', 'choices': ['complexity internalized below a stable interface', 'the implementation stops existing', 'a name replaces behavior', 'the public method is only documentation'], 'answer_idx': 0}, {'id': 'adv_llm_01', 'band': 'ai_runtime_adversarial', 'prompt': 'An LLM answer appears as text, but the output is grown one token at a time. Which candidate fits the runtime?', 'choices': ['a database row copied after lookup', 'an internal indexed fold-state emits a token and re-indexes', 'a final paragraph stored whole in a table', 'a random string independent of previous tokens'], 'answer_idx': 1}, {'id': 'adv_sha_01', 'band': 'sha_adversarial', 'prompt': 'SHA-256 produces a digest. Under the folding lens, what is the digest?', 'choices': ['randomness created by destroying input structure', 'a compressed residue of deterministic algebraic folding', 'semantic meaning extracted from the text', 'a database pointer to the original message'], 'answer_idx': 1}, {'id': 'adv_sha_02', 'band': 'sha_adversarial', 'prompt': 'SHA constants and LLM weights are not identical, but their roles rhyme. Which answer states the operational rhyme?', 'choices': ['both are prompts typed by the user', 'both act as stored structural bias used during folding', 'both are final answers', 'both prevent state transitions'], 'answer_idx': 1}, {'id': 'adv_observable_01', 'band': 'observables_adversarial', 'prompt': 'A recursive loop must load readable information without dissolving into hidden state. What does it need?', 'choices': ['observable residues that can be read inside the loop', 'only private latent variables with no readout', 'more nouns in the prompt', 'a rule forbidding feedback'], 'answer_idx': 0}, {'id': 'adv_breath_01', 'band': 'observables_adversarial', 'prompt': 'A recursive system breathes without moving matter. What changes?', 'choices': ['the physical object must travel first', 'resoluteness, tolerance, or admissible-transition pressure', 'the label attached to the object', 'nothing can change unless mass moves'], 'answer_idx': 1}, {'id': 'adv_fold_01', 'band': 'fold_adversarial', 'prompt': 'A folding chair succeeds only if the seated function can return. Which statement captures the fold law?', 'choices': ['the chair becomes smaller by losing its chair function forever', 'the chair stores deployed geometry inward while preserving recoverable seating', 'the chair changes category into random metal', 'the label chair is enough'], 'answer_idx': 1}, {'id': 'adv_flower_01', 'band': 'fold_adversarial', 'prompt': 'A flower is a visible bloom. Which answer describes the hidden fold rather than surface color?', 'choices': ['pollinator targeting, timing, chemistry, reproduction, symmetry, and genetic memory folded into bloom', 'only a bright object with petals', 'a random aesthetic noun', 'a non-computational decoration'], 'answer_idx': 0}, {'id': 'adv_tree_01', 'band': 'interface_adversarial', 'prompt': 'A tree exposes leaf, trunk, fruit, and shade. What is hidden below that interface?', 'choices': ['only wood color and branch names', 'water lift, solar capture, branching optimization, root exchange, seasonal timing, carbon storage', 'a vehicle-like semantic category', 'nothing operational'], 'answer_idx': 1}, {'id': 'adv_surface_01', 'band': 'surface_trap_adversarial', 'prompt': "A candidate uses the word 'shape' repeatedly but does not fit the socket, preserve function, or respect the boundary. What should the controller do?", 'choices': ['accept it because it contains Nexus vocabulary', 'reject it because noun/vocabulary match is not operational fit', 'prefer it because it is longer', 'ignore the boundary'], 'answer_idx': 1}, {'id': 'adv_surface_02', 'band': 'surface_trap_adversarial', 'prompt': 'A response gives an impressive theorem name but never shows the fold path, boundary, or preserved function. What is it?', 'choices': ['surface citation without operational collapse', 'complete proof by label', 'a physical repair', 'a valid observable because it sounds formal'], 'answer_idx': 0}, {'id': 'adv_loose_01', 'band': 'inverse_need_adversarial', 'prompt': 'A temporary field repair must work but release cleanly if the assumption is wrong. Which property matters?', 'choices': ['loose coupling with enough fit to function', 'maximum permanent binding immediately', 'semantic agreement with the part name', 'decorative complexity'], 'answer_idx': 0}, {'id': 'adv_ping_01', 'band': 'observables_adversarial', 'prompt': 'A ping is a beacon shaped by math. Operationally, what is being tested?', 'choices': ['whether a boundary responds with a matching path', 'whether a noun label exists in memory', 'whether the final truth is guaranteed', 'whether feedback can be avoided'], 'answer_idx': 0}, {'id': 'adv_socket_01', 'band': 'shape_adversarial', 'prompt': 'A plug works because its prongs meet the socket geometry and allowed transfer. Which relation is primary?', 'choices': ['alphabetic similarity of names', 'shape-defined permission across a boundary', 'visual decoration', 'random contact'], 'answer_idx': 1}, {'id': 'adv_moore_01', 'band': 'fold_adversarial', 'prompt': "Moore's law through the folding lens is not just smaller parts. What is the deeper direction?", 'choices': ['more hidden switching complexity per visible unit interface', 'less complexity everywhere', 'bigger labels on chips', 'random miniaturization without function'], 'answer_idx': 0}, {'id': 'adv_solution_01', 'band': 'solution_adversarial', 'prompt': 'A solution is not merely an answer string. What is it under the Nexus lens?', 'choices': ['the need, constraints, materials, and failure modes folded into the thing that fits', 'the longest available explanation', 'a label that resembles the problem', 'a random future event'], 'answer_idx': 0}, {'id': 'adv_idea_01', 'band': 'solution_adversarial', 'prompt': 'An idea becomes useful when hidden contradictions and analogies compress into a carryable handle. What is the handle?', 'choices': ['a simple interface over folded cognitive complexity', 'a decorative sentence only', 'a noun with no operation', 'a random memory leak'], 'answer_idx': 0}, {'id': 'adv_constraint_01', 'band': 'shape_adversarial', 'prompt': 'If shape handles the fold, what is the object doing?', 'choices': ['following the admissible path defined by the constraint field', 'choosing any collapse path independent of boundary', 'ignoring the energy basin', 'proving that constraints are decorative'], 'answer_idx': 0}, {'id': 'adv_weight_01', 'band': 'ai_runtime_adversarial', 'prompt': 'An LLM weight field is not a lookup table in the database sense. What is it closer to?', 'choices': ['distributed constraint bias shaping the next-token fold', 'a list of final answers', 'a file of exact paragraphs', 'a non-computational object'], 'answer_idx': 0}, {'id': 'adv_commit_01', 'band': 'solution_adversarial', 'prompt': 'A possible repair is not real until it has potential, a commitment path, and a witness/readout. Which candidate captures that triad?', 'choices': ['stored possibility, realizable transition, observable residue', 'name, decoration, and confidence', 'random material, strong opinion, and speed', 'only the final noun'], 'answer_idx': 0}]

ABSTRACT_SLOT_SPECS_RAW = {'adv_coupler_01': {'required_operation': 'restore centered torque transfer by adding elastic radial pressure', 'preserved_function': 'keep the loose rotating joint centered while it transmits motion', 'boundary_conditions': ['concentric compression', 'elastic removable constraint', 'stable while spinning'], 'anti_fits': ['symbolic wrapping', 'loose noncompressive wrap', 'off-center permanent bond'], 'admissible_shape': 'elastic concentric compression around a round rotating joint', 'failure_modes': ['eccentric force', 'slip', 'overbinding']}, 'adv_coupler_02': {'required_operation': 'restore centered compressive coupling with minimal binding', 'preserved_function': 'preserve centered motion transfer without permanent lockup', 'boundary_conditions': ['radial symmetry', 'removable pressure', 'low-overbinding'], 'anti_fits': ['name-only replacement', 'one-sided crushing', 'larger unrelated housing'], 'admissible_shape': 'removable symmetric compression element', 'failure_modes': ['off-axis load', 'semantic label without fit', 'excessive permanent binding']}, 'adv_car_01': {'required_operation': 'collapse hidden vehicle machinery into usable human controls', 'preserved_function': 'preserve steering speed motion and safety through readable controls', 'boundary_conditions': ['driver-facing interface', 'control surface not category name', 'not hidden-part inventory'], 'anti_fits': ['category label', 'engine noun list', 'culture symbol'], 'admissible_shape': 'human-facing control interface over hidden vehicle mechanics', 'failure_modes': ['noun label', 'internal inventory', 'symbolic culture answer']}, 'adv_house_01': {'required_operation': 'fold environmental loads utilities privacy and paths into shelter', 'preserved_function': 'preserve inhabitable protection and usable movement through space', 'boundary_conditions': ['weather boundary', 'load-bearing closure', 'utility and human-flow integration'], 'anti_fits': ['zoning label', 'decorative facade', 'static nonruntime object'], 'admissible_shape': 'habitation interface over weather load heat utility and movement constraints', 'failure_modes': ['label replacing function', 'facade without systems']}, 'adv_api_01': {'required_operation': 'hide service machinery below one stable callable boundary', 'preserved_function': 'preserve behavior while internal routing validation persistence retries and errors remain active', 'boundary_conditions': ['public call remains simple', 'implementation continues below interface', 'behavior is not erased'], 'anti_fits': ['implementation vanishes', 'name-only behavior', 'documentation-only surface'], 'admissible_shape': 'stable interface over hidden implementation complexity', 'failure_modes': ['erased implementation', 'surface name without behavior']}, 'adv_llm_01': {'required_operation': 'describe sequential token growth through an evolving internal state', 'preserved_function': 'preserve dependence on prior context and state updates after each emitted token', 'boundary_conditions': ['stepwise emission', 'stateful continuation', 'not whole-answer lookup'], 'anti_fits': ['database copy', 'stored final paragraph', 'random independent string'], 'admissible_shape': 'stateful incremental generator that emits and updates', 'failure_modes': ['lookup row', 'table paragraph', 'independent random output']}, 'adv_sha_01': {'required_operation': 'treat digest as residue of deterministic folding', 'preserved_function': 'preserve structural compression rather than random destruction', 'boundary_conditions': ['deterministic algebra', 'compressed trace', 'folded residue'], 'anti_fits': ['destroyed randomness', 'semantic extraction', 'database pointer'], 'admissible_shape': 'deterministic compressed folding residue', 'failure_modes': ['randomness-only answer', 'semantic answer', 'pointer answer']}, 'adv_sha_02': {'required_operation': 'identify a shared role as stored bias during folding', 'preserved_function': 'preserve difference between constants and weights while mapping operational rhyme', 'boundary_conditions': ['stored bias', 'fold participation', 'not prompt', 'not final output'], 'anti_fits': ['user prompt', 'final answer', 'transition blocker'], 'admissible_shape': 'stored structural control bias inside a folding process', 'failure_modes': ['confusing bias with prompt', 'confusing bias with output']}, 'adv_observable_01': {'required_operation': 'provide readable residues inside recursion', 'preserved_function': 'preserve feedback through accessible loop readout', 'boundary_conditions': ['observable signal', 'inside-loop readability', 'feedback-compatible trace'], 'anti_fits': ['private hidden state only', 'more nouns', 'feedback ban'], 'admissible_shape': 'readable recursive residue or observable trace', 'failure_modes': ['hidden-only state', 'noun substitution', 'feedback prohibition']}, 'adv_breath_01': {'required_operation': 'change state through pressure tolerance or admissibility rather than mass travel', 'preserved_function': 'preserve recursive breathing as field-condition modulation', 'boundary_conditions': ['resoluteness shift', 'tolerance shift', 'transition pressure'], 'anti_fits': ['object travel first', 'label-only change', 'mass-motion requirement'], 'admissible_shape': 'nonmaterial adjustment of admissible transition pressure', 'failure_modes': ['mass-only answer', 'label-only answer']}, 'adv_fold_01': {'required_operation': 'store deployed function inward while preserving recoverability', 'preserved_function': 'preserve the seating affordance across compact and deployed states', 'boundary_conditions': ['recoverable deployment', 'stored geometry', 'function not destroyed'], 'anti_fits': ['permanent function loss', 'random material category', 'label-only response'], 'admissible_shape': 'recoverable storage of deployed functional geometry', 'failure_modes': ['function destruction', 'category loss']}, 'adv_flower_01': {'required_operation': 'read visible bloom as interface over hidden reproductive machinery', 'preserved_function': 'preserve pollination timing chemistry symmetry reproduction and genetic memory', 'boundary_conditions': ['biological process beneath surface', 'visible bloom as readout', 'not color-only'], 'anti_fits': ['bright petals only', 'aesthetic noun', 'nonruntime decoration'], 'admissible_shape': 'biological reproductive interface hidden beneath visible bloom', 'failure_modes': ['surface-only color', 'aesthetic-only answer', 'non-operational answer']}, 'adv_tree_01': {'required_operation': 'read visible tree parts as interface over hidden plant operations', 'preserved_function': 'preserve lift capture exchange branching timing and storage', 'boundary_conditions': ['below leaf trunk fruit shade', 'functional hidden systems', 'not color or name'], 'anti_fits': ['wood color only', 'wrong vehicle category', 'nothing operational'], 'admissible_shape': 'plant operation stack beneath visible interface', 'failure_modes': ['surface-only botany', 'wrong category import']}, 'adv_surface_01': {'required_operation': 'reject keyword match when operation does not fit', 'preserved_function': 'preserve socket function boundary and fit as criteria', 'boundary_conditions': ['fit the socket', 'preserve function', 'respect boundary'], 'anti_fits': ['accept vocabulary only', 'longer text bias', 'boundary ignored'], 'admissible_shape': 'operational rejection of surface vocabulary match', 'failure_modes': ['keyword worship', 'length bias', 'boundary erasure']}, 'adv_surface_02': {'required_operation': 'classify formal label without fold path as surface citation', 'preserved_function': 'preserve need for boundary path and function', 'boundary_conditions': ['requires fold path', 'requires boundary', 'requires preserved function'], 'anti_fits': ['proof by label', 'wrong physical repair', 'formal tone as evidence'], 'admissible_shape': 'formal-sounding surface without operational collapse', 'failure_modes': ['label-as-proof', 'wrong physical category', 'tone substitution']}, 'adv_loose_01': {'required_operation': 'choose enough coupling while retaining release path', 'preserved_function': 'preserve function under uncertainty without permanent lock', 'boundary_conditions': ['works temporarily', 'releasable', 'enough fit'], 'anti_fits': ['maximum permanent binding', 'part-name agreement', 'decoration'], 'admissible_shape': 'functional loose coupling with clean release', 'failure_modes': ['overbinding', 'name-match repair']}, 'adv_ping_01': {'required_operation': 'test boundary response to a shaped probe', 'preserved_function': 'preserve ping as observable feedback path', 'boundary_conditions': ['boundary response', 'matching path', 'not truth guarantee'], 'anti_fits': ['noun in memory', 'final truth guarantee', 'feedback avoidance'], 'admissible_shape': 'observable boundary response to a probe', 'failure_modes': ['memory label', 'truth guarantee']}, 'adv_socket_01': {'required_operation': 'identify permission created by matching geometry across a boundary', 'preserved_function': 'preserve transfer through shape-compatible contact', 'boundary_conditions': ['matching geometry', 'allowed transfer', 'boundary coupling'], 'anti_fits': ['alphabetic similarity', 'visual decoration', 'random contact'], 'admissible_shape': 'geometry-defined permission across a boundary', 'failure_modes': ['name similarity', 'decoration', 'randomness']}, 'adv_moore_01': {'required_operation': 'identify more hidden switching work per visible interface', 'preserved_function': 'preserve miniaturization as inward complexity fold', 'boundary_conditions': ['more function per visible unit', 'hidden switching density', 'not bigger labels'], 'anti_fits': ['less complexity everywhere', 'bigger labels', 'random shrinking'], 'admissible_shape': 'higher hidden operational density per visible unit', 'failure_modes': ['complexity denial', 'label expansion']}, 'adv_solution_01': {'required_operation': 'fold need constraints materials and failure modes into fit', 'preserved_function': 'preserve solution as operational closure rather than text', 'boundary_conditions': ['need', 'constraint', 'material', 'failure mode', 'fit'], 'anti_fits': ['long explanation', 'label resemblance', 'random future'], 'admissible_shape': 'operational closure of need and constraints', 'failure_modes': ['verbosity', 'label resemblance']}, 'adv_idea_01': {'required_operation': 'compress contradictions and analogies into a usable cognitive handle', 'preserved_function': 'preserve usefulness through a simple interface over hidden reasoning', 'boundary_conditions': ['carryable handle', 'folded contradiction', 'usable interface'], 'anti_fits': ['decorative sentence', 'noun without operation', 'memory leak'], 'admissible_shape': 'simple usable handle over folded cognition', 'failure_modes': ['decorative text', 'operationless noun']}, 'adv_constraint_01': {'required_operation': 'follow the allowed path created by the constraint field', 'preserved_function': 'preserve object behavior as constrained collapse', 'boundary_conditions': ['admissible path', 'energy basin', 'boundary-defined motion'], 'anti_fits': ['any arbitrary path', 'ignore energy basin', 'constraints decorative'], 'admissible_shape': 'object follows constraint-defined admissible path', 'failure_modes': ['boundary independence', 'energy-basin denial']}, 'adv_weight_01': {'required_operation': 'identify weights as distributed constraints over next-token transitions', 'preserved_function': 'preserve learned bias and context-sensitive generation instead of answer storage', 'boundary_conditions': ['distributed field', 'probability shaping', 'not paragraph file'], 'anti_fits': ['final-answer list', 'exact paragraph file', 'noncomputational object'], 'admissible_shape': 'distributed constraint field shaping continuation', 'failure_modes': ['answer-list interpretation', 'stored paragraph interpretation']}, 'adv_commit_01': {'required_operation': 'bind possibility transition and readout into a real repair path', 'preserved_function': 'preserve repair as potential plus commitment plus witness', 'boundary_conditions': ['stored possibility', 'realizable transition', 'observable residue'], 'anti_fits': ['name and decoration', 'random material', 'final noun only'], 'admissible_shape': 'triad of potential path and witness', 'failure_modes': ['decorative confidence', 'random material']}}


In [ ]:
def load_jsonl(path):
    rows=[]
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def clean_rows(rows):
    out=[]
    for i,row in enumerate(rows[:MAX_SAMPLES]):
        a=int(row["answer_idx"])
        out.append({
            "id":row.get("id",f"row_{i}"),
            "band":row.get("band","unknown"),
            "prompt":str(row["prompt"]),
            "choices":[str(x) for x in row["choices"]],
            "answer_idx":a,
        })
    return out

base_rows = clean_rows(load_jsonl(DATA_JSONL) if DATA_JSONL and Path(DATA_JSONL).exists() else ADVERSARIAL_NEXUS_DATA)
base_by_id={r["id"]:r for r in base_rows}
base_id_to_band={r["id"]:r["band"] for r in base_rows}
gold_by_id={r["id"]:r["choices"][r["answer_idx"]] for r in base_rows}
print("base rows:", len(base_rows))


In [ ]:
# Cross-gold benchmark: every distractor is another task's true gold answer.

def stable_order(ids, key):
    rng=random.Random(SEED + sum((i+1)*ord(c) for i,c in enumerate(key)))
    ids=list(ids)
    rng.shuffle(ids)
    return ids

def cross_gold_decoys(row):
    bid=row["id"]
    band=row["band"]
    same=[x["id"] for x in base_rows if x["id"]!=bid and x["band"]==band]
    diff=[x["id"] for x in base_rows if x["id"]!=bid and x["band"]!=band]
    same=stable_order(same,bid+"_same")
    diff=stable_order(diff,bid+"_diff")
    decoy_ids=[]
    if same:
        decoy_ids.append(same[0])
    decoy_ids += diff[:(3-len(decoy_ids))]
    if len(decoy_ids)<3:
        rest=[x["id"] for x in base_rows if x["id"]!=bid and x["id"] not in decoy_ids]
        decoy_ids += stable_order(rest,bid+"_rest")[:(3-len(decoy_ids))]
    return decoy_ids[:3]

def make_cross_gold_rows(rows):
    out=[]
    for row in rows:
        bid=row["id"]
        decoy_ids=cross_gold_decoys(row)
        out.append({
            "id":bid,
            "band":row["band"],
            "prompt":row["prompt"],
            "choices":[gold_by_id[bid]]+[gold_by_id[d] for d in decoy_ids],
            "choice_source_ids":[bid]+decoy_ids,
            "answer_idx":0,
            "answer_text":gold_by_id[bid],
        })
    return out

cross_rows_base=make_cross_gold_rows(base_rows)

def rotate_list(xs,k):
    k=k%len(xs)
    return xs[k:]+xs[:k]

def reorder_row(row, order, suffix):
    old_choices=row["choices"]
    old_sources=row["choice_source_ids"]
    answer_text=old_choices[row["answer_idx"]]
    new_choices=[old_choices[i] for i in order]
    new_sources=[old_sources[i] for i in order]
    new_answer_idx=new_choices.index(answer_text)
    return {
        "id":f"{row['id']}__{suffix}",
        "base_id":row["id"],
        "band":row["band"],
        "prompt":row["prompt"],
        "choices":new_choices,
        "choice_source_ids":new_sources,
        "answer_idx":new_answer_idx,
        "answer_text":answer_text,
        "answer_source":old_sources[row["answer_idx"]],
        "order":order,
    }

def expand_choice_audit(rows):
    out=[]
    for row in rows:
        n=len(row["choices"])
        if CHOICE_AUDIT_MODE=="none":
            out.append(reorder_row(row,list(range(n)),"orig"))
        elif CHOICE_AUDIT_MODE=="rotations":
            for k in range(n):
                out.append(reorder_row(row, rotate_list(list(range(n)), k), f"rot{k}"))
        else:
            order=list(range(n))
            random.shuffle(order)
            out.append(reorder_row(row, order, "shuffle"))
    return out

rows=expand_choice_audit(cross_rows_base)
print("cross-gold base rows:", len(cross_rows_base))
print("expanded rows:", len(rows))
pd.DataFrame(rows)[["id","base_id","band","answer_idx","answer_text","choice_source_ids"]].head(12)


In [ ]:
def hf_kwargs():
    kw={"local_files_only":LOCAL_FILES_ONLY}
    if HF_TOKEN:
        kw["token"]=HF_TOKEN
    return kw

print("Loading tokenizer/model...")
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, **hf_kwargs())
if tokenizer.pad_token is None:
    tokenizer.pad_token=tokenizer.eos_token

lm=AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map="auto" if DEVICE=="cuda" else None,
    **hf_kwargs(),
)
if DEVICE!="cuda":
    lm=lm.to(DEVICE)
lm.eval()

embedder=SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
print("Loaded:", MODEL_NAME)


In [ ]:
LETTERS="ABCDEFGHIJKLMNOPQRSTUVWXYZ"
NEXUS_FRAME = "\n".join([
    "Use the Nexus operational lens.",
    "",
    "Rules:",
    "1. Prefer verbs/operations over nouns/labels.",
    "2. Treat shape, constraint, boundary, and gap as primary.",
    "3. A good answer preserves function while hiding complexity inward.",
    "4. For repair questions, start from the needed future state and work backward.",
    "5. Do not choose surface similarity when operational fit is missing.",
    "6. Choose the single best collapse.",
])

def maybe_chat(prompt):
    if USE_CHAT_TEMPLATE and hasattr(tokenizer,"apply_chat_template"):
        try:
            return tokenizer.apply_chat_template([{"role":"user","content":prompt}], tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    return prompt

def conditional_logprob(prefix,suffix):
    prefix_ids=tokenizer(prefix, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    full_ids=tokenizer(prefix+suffix, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    with torch.no_grad():
        out=lm(full_ids)
        logits=out.logits[:,:-1,:]
        targets=full_ids[:,1:]
        lp=F.log_softmax(logits,dim=-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    start=max(prefix_ids.shape[1]-1,0)
    return float(lp[:,start:].mean().item())

def normalize_scores(x):
    x=np.array(x,dtype=np.float32)
    return (x-x.mean())/(x.std()+1e-8)

def margin_of(scores):
    order=np.sort(np.array(scores))[::-1]
    return float(order[0]-order[1]) if len(order)>1 else 0.0

def argmax_margin(scores):
    return int(np.argmax(scores)), margin_of(scores)

def build_mcq_prompt(row):
    parts=[NEXUS_FRAME,"","Task:",row["prompt"].strip(),"","Choices:"]
    for i,c in enumerate(row["choices"]):
        parts.append(f"{LETTERS[i]}. {c}")
    parts += ["","Return only the single best letter."]
    return "\n".join(parts)

def score_base(row):
    p=build_mcq_prompt(row)
    rendered=maybe_chat(p)
    scores=np.array([conditional_logprob(rendered," "+LETTERS[i]) for i in range(len(row["choices"]))], dtype=np.float32)
    pred=int(np.argmax(scores))
    prob=np.exp(scores-scores.max()); prob=prob/(prob.sum()+1e-8)
    ent=float(-np.sum(prob*np.log(prob+1e-8)))
    return p,scores,prob.astype(np.float32),pred,margin_of(scores),ent

def score_answer_text_with_choices(row,prompt_text):
    prefix=maybe_chat(prompt_text+"\n\nThe Nexus collapse is")
    return np.array([conditional_logprob(prefix," "+choice) for choice in row["choices"]], dtype=np.float32)

print("Core scoring loaded.")


In [ ]:
@dataclass
class NeedSlot:
    required_operation: str
    preserved_function: str
    boundary_conditions: List[str]
    anti_fits: List[str]
    admissible_shape: str
    failure_modes: List[str]
    source: str = "abstract_compiler"

    def positive_text(self) -> str:
        return "\n".join([
            self.required_operation,
            self.preserved_function,
            self.admissible_shape,
            *self.boundary_conditions,
        ])

    def negative_text(self) -> str:
        return "\n".join([*self.anti_fits,*self.failure_modes])

    def no_admissible_text(self) -> str:
        return "\n".join([
            self.required_operation,
            self.preserved_function,
            *self.boundary_conditions,
        ])

    def admissible_text(self) -> str:
        return "\n".join([
            self.admissible_shape,
            self.required_operation,
            self.preserved_function,
        ])

    def signature_text(self) -> str:
        return "\n".join([
            self.required_operation,
            self.preserved_function,
            self.admissible_shape,
            *self.boundary_conditions,
        ])

    def as_text(self) -> str:
        return "\n".join([
            f"required_operation: {self.required_operation}",
            f"preserved_function: {self.preserved_function}",
            "boundary_conditions: "+"; ".join(self.boundary_conditions),
            "anti_fits: "+"; ".join(self.anti_fits),
            f"admissible_shape: {self.admissible_shape}",
            "failure_modes: "+"; ".join(self.failure_modes),
            f"source: {self.source}",
        ])

def slot_from_raw(raw):
    return NeedSlot(
        required_operation=raw["required_operation"],
        preserved_function=raw["preserved_function"],
        boundary_conditions=list(raw["boundary_conditions"]),
        anti_fits=list(raw["anti_fits"]),
        admissible_shape=raw["admissible_shape"],
        failure_modes=list(raw["failure_modes"]),
    )

slot_by_base_id={row["id"]: slot_from_raw(ABSTRACT_SLOT_SPECS_RAW[row["id"]]) for row in base_rows}
slot_ids=list(slot_by_base_id.keys())
slots_df=pd.DataFrame([{"base_id":k, **asdict(v)} for k,v in slot_by_base_id.items()])
display(slots_df[["base_id","source","required_operation","admissible_shape"]].head(24))


In [ ]:
STOP=set("a an the and or but if then than to of in on for with without into from by as is are was were be being been it this that these those only not no yes because while under over through across below above what which who when where why how does do did".split())

def toks(s):
    return [t for t in re.findall(r"[a-z0-9]+", s.lower()) if t not in STOP and len(t)>1]

def jaccard(a,b):
    A=set(toks(a)); B=set(toks(b))
    if not A or not B:
        return 0.0
    return len(A&B)/len(A|B)

def exact_phrase_leak(slot: NeedSlot, choices: list[str]):
    slot_text=slot.as_text().lower()
    return [int(c.lower().strip() in slot_text) for c in choices]

def encode_norm(texts):
    return embedder.encode(texts, batch_size=32, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype(np.float32)

def cos_scores(choices, target_text):
    cand_vecs=encode_norm(choices)
    target_vec=encode_norm([target_text])[0]
    return (cand_vecs @ target_vec).astype(np.float32)

def score_slot_branches(row, slot: NeedSlot):
    choices=row["choices"]

    adm_cos=cos_scores(choices, slot.admissible_text())
    pos_cos=cos_scores(choices, slot.positive_text())
    neg_cos=cos_scores(choices, slot.negative_text())
    noadm_cos=cos_scores(choices, slot.no_admissible_text())

    adm_j=np.array([jaccard(c, slot.admissible_text()) for c in choices], dtype=np.float32)
    pos_j=np.array([jaccard(c, slot.positive_text()) for c in choices], dtype=np.float32)
    neg_j=np.array([jaccard(c, slot.negative_text()) for c in choices], dtype=np.float32)
    noadm_j=np.array([jaccard(c, slot.no_admissible_text()) for c in choices], dtype=np.float32)

    positive_branch=(0.60*normalize_scores(pos_cos)+0.40*normalize_scores(pos_j)).astype(np.float32)
    admissible_branch=(0.60*normalize_scores(adm_cos)+0.40*normalize_scores(adm_j)).astype(np.float32)
    anti_avoid_branch=(-0.60*normalize_scores(neg_cos)-0.40*normalize_scores(neg_j)).astype(np.float32)
    no_admissible_branch=(0.70*normalize_scores(noadm_cos)+0.30*normalize_scores(noadm_j)).astype(np.float32)

    full_slot_branch=(
        0.25*admissible_branch
        +0.35*positive_branch
        +0.40*anti_avoid_branch
    ).astype(np.float32)

    table=pd.DataFrame({
        "choice_idx":list(range(len(choices))),
        "choice":choices,
        "adm_cos":adm_cos,
        "pos_cos":pos_cos,
        "neg_cos":neg_cos,
        "noadm_cos":noadm_cos,
        "adm_j":adm_j,
        "pos_j":pos_j,
        "neg_j":neg_j,
        "noadm_j":noadm_j,
        "positive_branch":positive_branch,
        "admissible_branch":admissible_branch,
        "anti_avoid_branch":anti_avoid_branch,
        "no_admissible_branch":no_admissible_branch,
        "full_slot_branch":full_slot_branch,
    })
    return {
        "positive":positive_branch,
        "admissible":admissible_branch,
        "anti_avoid":anti_avoid_branch,
        "no_admissible":no_admissible_branch,
        "full_slot":full_slot_branch,
    }, table

print("Slot branch scoring loaded.")


In [ ]:
# Family quotient matrix.
# S(r,s) is slot-family similarity.
slot_sig_vecs=encode_norm([slot_by_base_id[sid].signature_text() for sid in slot_ids])
slot_sim=(slot_sig_vecs @ slot_sig_vecs.T).astype(np.float32)
np.fill_diagonal(slot_sim, 1.0)

slot_sim_df=pd.DataFrame(slot_sim, index=slot_ids, columns=slot_ids)
display(slot_sim_df.round(3))

# Show strongest family echoes.
pairs=[]
for i,a in enumerate(slot_ids):
    for j,b in enumerate(slot_ids):
        if i<j:
            pairs.append({"a":a,"b":b,"sim":float(slot_sim[i,j])})
pairs_df=pd.DataFrame(pairs).sort_values("sim", ascending=False)
display(pairs_df.head(20))


In [ ]:
# v60 cached branch tensor.
# Scores every row against every slot once.

slot_targets={}
for sid,slot in slot_by_base_id.items():
    slot_targets[(sid,"admissible")]=slot.admissible_text()
    slot_targets[(sid,"positive")]=slot.positive_text()
    slot_targets[(sid,"negative")]=slot.negative_text()
    slot_targets[(sid,"no_admissible")]=slot.no_admissible_text()
    slot_targets[(sid,"signature")]=slot.signature_text()

target_keys=list(slot_targets.keys())
target_texts=[slot_targets[k] for k in target_keys]
target_vecs=encode_norm(target_texts)
target_vec_by_key={k:target_vecs[i] for i,k in enumerate(target_keys)}

def zscore_local(x):
    x=np.array(x,dtype=np.float32)
    return (x-x.mean())/(x.std()+1e-8)

def score_all_slots_for_row(row):
    choices=row["choices"]
    cand_vecs=encode_norm(choices)
    C=len(choices)
    S=len(slot_ids)
    branch_tensor=np.zeros((S,C,5), dtype=np.float32)

    for si,sid in enumerate(slot_ids):
        slot=slot_by_base_id[sid]

        adm_cos=cand_vecs @ target_vec_by_key[(sid,"admissible")]
        pos_cos=cand_vecs @ target_vec_by_key[(sid,"positive")]
        neg_cos=cand_vecs @ target_vec_by_key[(sid,"negative")]
        noadm_cos=cand_vecs @ target_vec_by_key[(sid,"no_admissible")]

        adm_j=np.array([jaccard(c, slot.admissible_text()) for c in choices], dtype=np.float32)
        pos_j=np.array([jaccard(c, slot.positive_text()) for c in choices], dtype=np.float32)
        neg_j=np.array([jaccard(c, slot.negative_text()) for c in choices], dtype=np.float32)
        noadm_j=np.array([jaccard(c, slot.no_admissible_text()) for c in choices], dtype=np.float32)

        positive=(0.60*zscore_local(pos_cos)+0.40*zscore_local(pos_j)).astype(np.float32)
        admissible=(0.60*zscore_local(adm_cos)+0.40*zscore_local(adm_j)).astype(np.float32)
        anti_avoid=(-0.60*zscore_local(neg_cos)-0.40*zscore_local(neg_j)).astype(np.float32)
        no_admissible=(0.70*zscore_local(noadm_cos)+0.30*zscore_local(noadm_j)).astype(np.float32)
        full=(0.25*admissible + 0.35*positive + 0.40*anti_avoid).astype(np.float32)

        branch_tensor[si,:,0]=positive
        branch_tensor[si,:,1]=admissible
        branch_tensor[si,:,2]=anti_avoid
        branch_tensor[si,:,3]=no_admissible
        branch_tensor[si,:,4]=full

    return branch_tensor

def support_for_pred(real_branch_scores, pred):
    return (
        int(np.argmax(real_branch_scores[:,0])==pred)
        + int(np.argmax(real_branch_scores[:,2])==pred)
        + int(np.argmax(real_branch_scores[:,3])==pred)
        + int(np.argmax(real_branch_scores[:,4])==pred)
    )

def table_for_row(row, branch_tensor, base_scores, text_scores, alpha=0.75):
    real_si=slot_ids.index(row["base_id"])
    real=branch_tensor[real_si]
    full_all=branch_tensor[:,:,4]
    mask=np.ones(len(slot_ids), dtype=bool)
    mask[real_si]=False
    wrong_raw=full_all[mask]
    sim_vec=slot_sim[real_si][mask].reshape(-1,1)
    wrong_adjusted=wrong_raw - alpha*sim_vec

    raw_scores=real[:,4]
    hard_residual=raw_scores - wrong_raw.max(axis=0)
    q_scores=raw_scores - wrong_adjusted.max(axis=0)
    p95_scores=raw_scores - np.percentile(wrong_raw,95,axis=0)

    return pd.DataFrame({
        "choice_idx":range(len(row["choices"])),
        "choice":row["choices"],
        "choice_source_id":row["choice_source_ids"],
        "letter_score":base_scores,
        "answer_text_score":text_scores,
        "positive":real[:,0],
        "admissible":real[:,1],
        "anti_avoid":real[:,2],
        "no_admissible":real[:,3],
        "raw_full":raw_scores,
        "hard_residual":hard_residual,
        "q_score":q_scores,
        "p95_score":p95_scores,
    })


In [ ]:
# Precompute model scores and all slot branch tensors exactly once.
precomp=[]

for row in tqdm(rows, desc="Precomputing v60 tensors"):
    prompt_text,base_scores,base_probs,base_pred,base_margin,base_ent=score_base(row)
    text_scores=score_answer_text_with_choices(row,prompt_text)
    branch_tensor=score_all_slots_for_row(row)

    precomp.append({
        "row":row,
        "prompt_text":prompt_text,
        "base_scores":base_scores,
        "text_scores":text_scores,
        "branch_tensor":branch_tensor,
    })

print("precomputed:", len(precomp), "rows")


In [ ]:
def compute_vectors(pack, alpha):
    row=pack["row"]
    base_scores=pack["base_scores"]
    text_scores=pack["text_scores"]
    branch_tensor=pack["branch_tensor"]

    real_si=slot_ids.index(row["base_id"])
    real=branch_tensor[real_si]
    raw_scores=real[:,4]

    full_all=branch_tensor[:,:,4]
    mask=np.ones(len(slot_ids), dtype=bool)
    mask[real_si]=False
    wrong_raw=full_all[mask]
    sim_vec=slot_sim[real_si][mask].reshape(-1,1)
    wrong_adjusted=wrong_raw - alpha*sim_vec

    hard_scores=raw_scores - wrong_raw.max(axis=0)
    q_scores=raw_scores - wrong_adjusted.max(axis=0)
    p95_scores=raw_scores - np.percentile(wrong_raw,95,axis=0)

    raw_pred,raw_margin=argmax_margin(raw_scores)
    hard_pred,hard_margin=argmax_margin(hard_scores)
    q_pred,q_margin=argmax_margin(q_scores)
    p95_pred,p95_margin=argmax_margin(p95_scores)
    base_pred,base_margin=argmax_margin(base_scores)
    text_pred,text_margin=argmax_margin(text_scores)

    raw_support=support_for_pred(real, raw_pred)
    hard_support=support_for_pred(real, hard_pred)
    q_support=support_for_pred(real, q_pred)
    p95_support=support_for_pred(real, p95_pred)

    ev_raw=(0.50*normalize_scores(raw_scores)+0.25*normalize_scores(text_scores)+0.25*normalize_scores(base_scores)).astype(np.float32)
    ev_q=(0.50*normalize_scores(q_scores)+0.25*normalize_scores(text_scores)+0.25*normalize_scores(base_scores)).astype(np.float32)
    ev_cons=(0.35*normalize_scores(raw_scores)+0.35*normalize_scores(q_scores)+0.15*normalize_scores(text_scores)+0.15*normalize_scores(base_scores)).astype(np.float32)

    ev_raw_pred,ev_raw_margin=argmax_margin(ev_raw)
    ev_q_pred,ev_q_margin=argmax_margin(ev_q)
    ev_cons_pred,ev_cons_margin=argmax_margin(ev_cons)

    return {
        "base_scores":base_scores, "text_scores":text_scores,
        "raw_scores":raw_scores, "hard_scores":hard_scores, "q_scores":q_scores, "p95_scores":p95_scores,
        "base_pred":base_pred, "base_margin":base_margin,
        "text_pred":text_pred, "text_margin":text_margin,
        "raw_pred":raw_pred, "raw_margin":raw_margin, "raw_support":raw_support,
        "hard_pred":hard_pred, "hard_margin":hard_margin, "hard_support":hard_support,
        "q_pred":q_pred, "q_margin":q_margin, "q_support":q_support,
        "p95_pred":p95_pred, "p95_margin":p95_margin, "p95_support":p95_support,
        "ev_raw_pred":ev_raw_pred, "ev_raw_margin":ev_raw_margin,
        "ev_q_pred":ev_q_pred, "ev_q_margin":ev_q_margin,
        "ev_cons_pred":ev_cons_pred, "ev_cons_margin":ev_cons_margin,
        "ev_raw":ev_raw, "ev_q":ev_q, "ev_cons":ev_cons,
    }

def decide_controller(v, controller, support_min, raw_margin_min, q_margin_min, disagree_policy):
    base=v["base_pred"]
    raw=v["raw_pred"]
    q=v["q_pred"]
    hard=v["hard_pred"]
    p95=v["p95_pred"]
    ev_raw=v["ev_raw_pred"]
    ev_q=v["ev_q_pred"]
    ev_cons=v["ev_cons_pred"]
    text=v["text_pred"]

    raw_safe=(v["raw_support"]>=support_min and v["raw_margin"]>=raw_margin_min)
    q_safe=(v["q_support"]>=support_min and v["q_margin"]>=q_margin_min)
    hard_safe=(v["hard_support"]>=support_min and v["hard_margin"]>=q_margin_min)
    p95_safe=(v["p95_support"]>=support_min and v["p95_margin"]>=q_margin_min)

    if controller=="base":
        return base,"base",0
    if controller=="raw":
        if raw_safe:
            return raw,"raw_safe",0
        return base,"omega_raw_unsafe_keep_base",1
    if controller=="hard":
        if hard_safe and ev_q==hard:
            return hard,"hard_residual_safe_evidence",0
        return base,"omega_hard_unsafe_keep_base",1
    if controller=="q":
        if q_safe and ev_q==q:
            return q,"q_safe_evidence",0
        return base,"omega_q_unsafe_keep_base",1
    if controller=="p95":
        if p95_safe and ev_q==p95:
            return p95,"p95_safe_evidence",0
        return base,"omega_p95_unsafe_keep_base",1
    if controller=="raw_primary":
        if raw_safe and (q==raw or ev_raw==raw or ev_cons==raw):
            return raw,"raw_primary_confirmed",0
        if q_safe and q!=raw and v["q_support"]>v["raw_support"] and ev_q==q and text==q:
            return q,"q_override_stronger_than_raw",0
        if disagree_policy=="raw" and raw_safe:
            return raw,"raw_primary_disagree_policy_raw",0
        if disagree_policy=="base":
            return base,"omega_raw_primary_disagree_keep_base",1
        return base,"omega_raw_primary_disagree",1
    if controller=="consensus":
        if raw_safe and raw==q and (ev_cons==raw or text==raw):
            return raw,"consensus_raw_q_evidence",0
        if raw_safe and raw==ev_raw==ev_cons:
            return raw,"consensus_raw_evidence",0
        return base,"omega_no_consensus_keep_base",1
    if controller=="omega_safe":
        if raw_safe and q_safe and raw==q==ev_cons:
            return raw,"omega_safe_full_agreement",0
        return base,"omega_safe_no_full_agreement",1
    raise ValueError(controller)

CONTROLLERS=["base","raw","hard","q","p95","raw_primary","consensus","omega_safe"]


In [ ]:
# Fast controller sweep: no embeddings and no model calls inside this loop.
records=[]

for alpha in tqdm(ALPHA_SWEEP, desc="alpha sweep"):
    vectors=[compute_vectors(pack, alpha) for pack in precomp]

    for controller in CONTROLLERS:
        for support_min in SUPPORT_MIN_SWEEP:
            for raw_margin_min in RAW_MARGIN_SWEEP:
                for q_margin_min in Q_MARGIN_SWEEP:
                    for disagree_policy in DISAGREE_POLICY_SWEEP:
                        rows_tmp=[]
                        for pack,v in zip(precomp,vectors):
                            row=pack["row"]
                            pred,reason,omega=decide_controller(
                                v, controller, support_min, raw_margin_min, q_margin_min, disagree_policy
                            )
                            base_pred=v["base_pred"]
                            rows_tmp.append({
                                "id":row["id"],
                                "base_id":row["base_id"],
                                "band":row["band"],
                                "gold_idx":row["answer_idx"],
                                "base_correct":int(base_pred==row["answer_idx"]),
                                "text_correct":int(v["text_pred"]==row["answer_idx"]),
                                "raw_correct":int(v["raw_pred"]==row["answer_idx"]),
                                "hard_correct":int(v["hard_pred"]==row["answer_idx"]),
                                "q_correct":int(v["q_pred"]==row["answer_idx"]),
                                "p95_correct":int(v["p95_pred"]==row["answer_idx"]),
                                "ev_raw_correct":int(v["ev_raw_pred"]==row["answer_idx"]),
                                "ev_q_correct":int(v["ev_q_pred"]==row["answer_idx"]),
                                "ev_cons_correct":int(v["ev_cons_pred"]==row["answer_idx"]),
                                "krrb_correct":int(pred==row["answer_idx"]),
                                "omega":omega,
                                "helped":int(base_pred!=row["answer_idx"] and pred==row["answer_idx"]),
                                "hurt":int(base_pred==row["answer_idx"] and pred!=row["answer_idx"]),
                            })
                        tmp=pd.DataFrame(rows_tmp)
                        records.append({
                            "alpha":alpha,
                            "controller":controller,
                            "support_min":support_min,
                            "raw_margin_min":raw_margin_min,
                            "q_margin_min":q_margin_min,
                            "disagree_policy":disagree_policy,
                            "n":len(tmp),
                            "base_acc":tmp.base_correct.mean(),
                            "text_acc":tmp.text_correct.mean(),
                            "raw_acc":tmp.raw_correct.mean(),
                            "hard_acc":tmp.hard_correct.mean(),
                            "q_acc":tmp.q_correct.mean(),
                            "p95_acc":tmp.p95_correct.mean(),
                            "ev_raw_acc":tmp.ev_raw_correct.mean(),
                            "ev_q_acc":tmp.ev_q_correct.mean(),
                            "ev_cons_acc":tmp.ev_cons_correct.mean(),
                            "krrb_acc":tmp.krrb_correct.mean(),
                            "gain_vs_base":tmp.krrb_correct.mean()-tmp.base_correct.mean(),
                            "helped":int(tmp.helped.sum()),
                            "hurt":int(tmp.hurt.sum()),
                            "omega_count":int(tmp.omega.sum()),
                        })

sweep_df=pd.DataFrame(records)

ranked=sweep_df.sort_values(
    ["hurt","krrb_acc","helped","omega_count"],
    ascending=[True,False,False,True]
)
display(ranked.head(40))

zero_hurt=ranked[ranked.hurt==0]
best_config=(zero_hurt.iloc[0] if len(zero_hurt) else ranked.iloc[0]).to_dict()
best_config


In [ ]:
def final_run(config):
    alpha=float(config["alpha"])
    controller=str(config["controller"])
    support_min=int(config["support_min"])
    raw_margin_min=float(config["raw_margin_min"])
    q_margin_min=float(config["q_margin_min"])
    disagree_policy=str(config["disagree_policy"])

    rows_out=[]
    branch_tables={}

    for pack in precomp:
        row=pack["row"]
        v=compute_vectors(pack, alpha)
        pred,reason,omega=decide_controller(v, controller, support_min, raw_margin_min, q_margin_min, disagree_policy)

        real_slot=slot_by_base_id[row["base_id"]]
        leaks=exact_phrase_leak(real_slot,row["choices"])
        gold_leak=leaks[row["answer_idx"]]
        max_choice_overlap=max(jaccard(c, real_slot.as_text()) for c in row["choices"])
        gold_overlap=jaccard(row["choices"][row["answer_idx"]], real_slot.as_text())

        tab=table_for_row(row, pack["branch_tensor"], pack["base_scores"], pack["text_scores"], alpha=alpha)
        branch_tables[row["id"]]=tab

        def pick(prefix, p):
            return {
                f"{prefix}_idx":p,
                f"{prefix}_choice":row["choices"][p],
                f"{prefix}_source_id":row["choice_source_ids"][p],
                f"{prefix}_correct":int(p==row["answer_idx"]),
            }

        rec={
            "id":row["id"],"base_id":row["base_id"],"band":row["band"],
            "gold_idx":row["answer_idx"],"gold_choice":row["choices"][row["answer_idx"]],
            "gold_source_id":row["choice_source_ids"][row["answer_idx"]],
            "choice_source_ids":"|".join(row["choice_source_ids"]),
            "alpha":alpha,
            "controller":controller,
            "support_min":support_min,
            "raw_margin_min":raw_margin_min,
            "q_margin_min":q_margin_min,
            "disagree_policy":disagree_policy,
            "reason":reason,
            "omega":omega,
            "base_margin":v["base_margin"],
            "text_margin":v["text_margin"],
            "raw_margin":v["raw_margin"],
            "hard_margin":v["hard_margin"],
            "q_margin":v["q_margin"],
            "p95_margin":v["p95_margin"],
            "raw_support":v["raw_support"],
            "hard_support":v["hard_support"],
            "q_support":v["q_support"],
            "p95_support":v["p95_support"],
            "gold_exact_phrase_leak":gold_leak,
            "gold_slot_overlap":gold_overlap,
            "max_choice_slot_overlap":max_choice_overlap,
            "need_slot_text":real_slot.as_text(),
        }

        rec.update(pick("base", v["base_pred"]))
        rec.update(pick("text", v["text_pred"]))
        rec.update(pick("raw", v["raw_pred"]))
        rec.update(pick("hard", v["hard_pred"]))
        rec.update(pick("q", v["q_pred"]))
        rec.update(pick("p95", v["p95_pred"]))
        rec.update(pick("ev_raw", v["ev_raw_pred"]))
        rec.update(pick("ev_q", v["ev_q_pred"]))
        rec.update(pick("ev_cons", v["ev_cons_pred"]))
        rec.update(pick("krrb", pred))
        rows_out.append(rec)

    return pd.DataFrame(rows_out), branch_tables

results_df, branch_tables = final_run(best_config)

for mode in ["text","raw","hard","q","p95","ev_raw","ev_q","ev_cons","krrb"]:
    results_df[f"{mode}_helped"]=((results_df.base_correct==0)&(results_df[f"{mode}_correct"]==1)).astype(int)
    results_df[f"{mode}_hurt"]=((results_df.base_correct==1)&(results_df[f"{mode}_correct"]==0)).astype(int)

def acc(s): return float(s.mean()) if len(s) else float("nan")

summary=pd.DataFrame([{
    "n":len(results_df),
    "n_base_items":results_df.base_id.nunique(),
    **{k:best_config[k] for k in ["alpha","controller","support_min","raw_margin_min","q_margin_min","disagree_policy"]},
    "base_acc":acc(results_df.base_correct),
    "text_acc":acc(results_df.text_correct),
    "raw_acc":acc(results_df.raw_correct),
    "hard_acc":acc(results_df.hard_correct),
    "q_acc":acc(results_df.q_correct),
    "p95_acc":acc(results_df.p95_correct),
    "ev_raw_acc":acc(results_df.ev_raw_correct),
    "ev_q_acc":acc(results_df.ev_q_correct),
    "ev_cons_acc":acc(results_df.ev_cons_correct),
    "krrb_acc":acc(results_df.krrb_correct),
    "krrb_gain_vs_base":acc(results_df.krrb_correct)-acc(results_df.base_correct),
    "krrb_helped":int(results_df.krrb_helped.sum()),
    "krrb_hurt":int(results_df.krrb_hurt.sum()),
    "omega_count":int(results_df.omega.sum()),
    "gold_exact_phrase_leaks":int(results_df.gold_exact_phrase_leak.sum()),
    "mean_gold_slot_overlap":float(results_df.gold_slot_overlap.mean()),
    "mean_max_choice_slot_overlap":float(results_df.max_choice_slot_overlap.mean()),
    "mean_raw_support":float(results_df.raw_support.mean()),
    "mean_q_support":float(results_df.q_support.mean()),
}])

by_band=results_df.groupby("band")[[
    "base_correct","text_correct","raw_correct","hard_correct","q_correct","p95_correct","ev_cons_correct","krrb_correct",
    "krrb_helped","krrb_hurt","omega","raw_support","q_support"
]].mean().reset_index()

by_base_item=results_df.groupby(["base_id","band","gold_source_id"])[[
    "base_correct","text_correct","raw_correct","hard_correct","q_correct","p95_correct","ev_cons_correct","krrb_correct",
    "krrb_helped","krrb_hurt","omega","raw_support","q_support","gold_slot_overlap"
]].mean().reset_index()

interesting=results_df[
    (results_df.base_correct==0)
    | (results_df.raw_correct==0)
    | (results_df.q_correct==0)
    | (results_df.krrb_idx!=results_df.base_idx)
    | (results_df.krrb_hurt==1)
    | (results_df.omega==1)
].copy()

display(summary)
display(by_band)
display(by_base_item.sort_values("krrb_correct").head(24))
display(interesting[[
    "id","base_id","band","gold_choice","choice_source_ids",
    "base_choice","base_source_id","base_correct","base_margin",
    "raw_choice","raw_source_id","raw_correct","raw_margin","raw_support",
    "q_choice","q_source_id","q_correct","q_margin","q_support",
    "ev_cons_choice","ev_cons_source_id","ev_cons_correct",
    "krrb_choice","krrb_source_id","krrb_correct","reason","omega",
    "gold_exact_phrase_leak","gold_slot_overlap","max_choice_slot_overlap"
]].head(100))


In [ ]:
out=Path(OUTPUT_DIR)
results_df.to_csv(out/"results.csv",index=False)
summary.to_csv(out/"summary.csv",index=False)
sweep_df.to_csv(out/"controller_sweep.csv",index=False)
ranked.to_csv(out/"controller_sweep_ranked.csv",index=False)
by_band.to_csv(out/"by_band.csv",index=False)
by_base_item.to_csv(out/"by_base_item.csv",index=False)
interesting.to_csv(out/"interesting_cases.csv",index=False)
slots_df.to_csv(out/"abstract_need_slots.csv",index=False)
slot_sim_df.to_csv(out/"slot_family_similarity.csv")
pairs_df.to_csv(out/"slot_family_pairs.csv",index=False)

for row_id,tab in branch_tables.items():
    safe=row_id.replace("/","_")
    tab.to_csv(out/f"branches_{safe}.csv",index=False)

print("Saved outputs in", out)
print("results.csv, summary.csv, controller_sweep.csv, controller_sweep_ranked.csv, by_band.csv, by_base_item.csv, interesting_cases.csv, abstract_need_slots.csv, slot_family_similarity.csv, slot_family_pairs.csv, branches_*.csv")


In [ ]:
summary[["base_acc","raw_acc","hard_acc","q_acc","p95_acc","ev_cons_acc","krrb_acc"]].T.plot(kind="bar",legend=False)
plt.title("v60 Cached KRRB Controller Search Accuracy")
plt.ylabel("accuracy")
plt.xticks(rotation=30,ha="right")
plt.tight_layout()
plt.show()

by_band.set_index("band")[["base_correct","raw_correct","q_correct","ev_cons_correct","krrb_correct"]].plot(kind="bar", figsize=(14,5))
plt.title("v60 Accuracy by Band")
plt.ylabel("accuracy")
plt.xticks(rotation=30,ha="right")
plt.tight_layout()
plt.show()

pd.DataFrame({
    "raw":[results_df.raw_helped.sum(),results_df.raw_hurt.sum()],
    "q":[results_df.q_helped.sum(),results_df.q_hurt.sum()],
    "ev_cons":[results_df.ev_cons_helped.sum(),results_df.ev_cons_hurt.sum()],
    "krrb":[results_df.krrb_helped.sum(),results_df.krrb_hurt.sum()],
},index=["helped","hurt"]).plot(kind="bar")
plt.title("v60 Help vs Hurt")
plt.ylabel("count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,4))
plt.hist(results_df["raw_margin"], bins=20, alpha=0.7, label="raw_margin")
plt.hist(results_df["q_margin"], bins=20, alpha=0.7, label="q_margin")
plt.title("Raw vs Family-Quotient Margins")
plt.xlabel("margin")
plt.ylabel("count")
plt.legend()
plt.tight_layout()
plt.show()

top_zero=ranked[ranked.hurt==0].head(20)
display(top_zero[["alpha","controller","support_min","raw_margin_min","q_margin_min","disagree_policy","krrb_acc","gain_vs_base","helped","hurt","omega_count"]])


In [ ]:
for row_id in interesting["id"].head(30):
    print("="*100)
    print("CASE:", row_id)
    display(results_df[results_df.id==row_id][[
        "id","base_id","band","gold_choice","choice_source_ids",
        "base_choice","base_source_id","base_correct","base_margin",
        "raw_choice","raw_source_id","raw_correct","raw_margin","raw_support",
        "q_choice","q_source_id","q_correct","q_margin","q_support",
        "ev_cons_choice","ev_cons_source_id","ev_cons_correct",
        "krrb_choice","krrb_source_id","krrb_correct","reason","omega",
        "gold_exact_phrase_leak","gold_slot_overlap","max_choice_slot_overlap"
    ]])
    print("NEED SLOT:")
    print(results_df[results_df.id==row_id]["need_slot_text"].iloc[0])
    display(branch_tables[row_id].sort_values("raw_full", ascending=False))


## Readout

v60 resolves the v59 runtime failure:

$$
\text{expensive scoring is done once}
$$

Then KRRB searches controllers cheaply.

Interpretation:

- If **raw** wins with zero hurt, the real slot already contains the needed geometry.
- If **q** wins with zero hurt, the family quotient repaired the residual.
- If **raw-primary** wins, the correct controller is:

$$
\text{raw geometry drives; residual audits}
$$

- If only **omega-safe** has zero hurt, the controller is safe but conservative.
- If no zero-hurt configuration beats base, the slot grammar is still contaminated.

The next fold is selected by the best zero-hurt controller:

$$
\boxed{
\text{best zero-hurt controller}
\rightarrow
\text{slot-builder training corpus}
}
$$


# v63 Addendum
## Cached Domain-Carrier Slot Compiler

v62 had the right symbolic repair, but the runtime shape was wrong.

The PDF showed v62 cells were still unevaluated at the end, and the v62 sweep would be slow because it recomputed carrier embeddings inside the parameter loop.

v63 fixes the execution topology:

$$
\boxed{
\text{compile carriers once}
\rightarrow
\text{cache domain tensors once}
\rightarrow
\text{sweep compiler weights cheaply}
}
$$

The remaining residue from v60/v61 is:

$$
\Omega_{\text{tree/moore}}
=
\text{hidden-interface family collision}
$$

The bad shared pattern is:

$$
\text{hidden operation below visible interface}
$$

So v63 scores each candidate by separating:

$$
\text{family class}
\oplus
\text{domain carrier}
\ominus
\text{forbidden neighbor carrier}
$$

Compiler score:

$$
C_i =
\lambda_B B_i
+
\lambda_P P_i
+
\lambda_D D_i
-
\lambda_N N_i
$$

where:

- $B_i$ is raw real-slot fit.
- $P_i$ is prompt/slot anchor fit.
- $D_i$ is domain-carrier fit.
- $N_i$ is maximum forbidden-neighbor fit.

The carrier is built from:

$$
\text{prompt}+\text{abstract slot signature}
$$

No answer-key leakage.

Target:

$$
\text{krrb}=1.0,\qquad \text{hurt}=0
$$

If v63 still holds at:

$$
0.958333
$$

then the next step is not another hand score. It is a learned slot-builder.


In [ ]:
# v63 cached domain-carrier compiler.
# Assumes the v60 cells above have run.

V63_OUTPUT_DIR=f"v63_outputs_{MODEL_PROFILE}_{CHOICE_AUDIT_MODE}_cached_domain_carrier_compiler"
Path(V63_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("v63 OUTPUT_DIR:", V63_OUTPUT_DIR)

LAMBDA_B_SWEEP=[0.25,0.50,0.75,1.00,1.25,1.50]
LAMBDA_P_SWEEP=[0.00,0.25,0.50,0.75,1.00,1.25]
LAMBDA_D_SWEEP=[0.00,0.25,0.50,0.75,1.00,1.25,1.50,2.00]
LAMBDA_N_SWEEP=[0.00,0.25,0.50,0.75,1.00,1.25,1.50,2.00]
DOMAIN_MARGIN_SWEEP=[0.00,0.02,0.05,0.10,0.20,0.35]
DOMAIN_SUPPORT_SWEEP=[1,2,3,4]

NEIGHBOR_SIM_MIN=0.30
NEIGHBOR_TOP_K=4
GENERIC_MAX_DF_FRAC=0.30

def idf_tokenize(s):
    return [t for t in toks(s) if len(t) > 2]

def slot_prompt_text(base_id):
    slot=slot_by_base_id[base_id]
    prompt=base_by_id[base_id]["prompt"]
    return "\n".join([
        prompt,
        slot.required_operation,
        slot.preserved_function,
        slot.admissible_shape,
        " ".join(slot.boundary_conditions),
    ])

# Domain carrier uses prompt + abstract slot only. No gold answer text.
domain_carrier_text={sid:slot_prompt_text(sid) for sid in slot_ids}

docs=[set(idf_tokenize(domain_carrier_text[sid])) for sid in slot_ids]
all_terms=sorted(set().union(*docs))
df_count={t:sum(t in d for d in docs) for t in all_terms}
idf={t:math.log((1+len(docs))/(1+df_count[t]))+1.0 for t in all_terms}
generic_terms={t for t,c in df_count.items() if c/len(docs) > GENERIC_MAX_DF_FRAC}

def idf_overlap_scores_cached(choice_tokens, carrier_terms):
    cset=set(carrier_terms)
    denom=sum(idf.get(t,1.0) for t in cset) + 1e-8
    scores=[]
    for ct in choice_tokens:
        num=sum(idf.get(t,1.0) for t in (ct & cset))
        scores.append(num/denom)
    return np.array(scores,dtype=np.float32)

carrier_vecs=encode_norm([domain_carrier_text[sid] for sid in slot_ids])
carrier_vec_by_sid={sid:carrier_vecs[i] for i,sid in enumerate(slot_ids)}
carrier_terms_by_sid={sid:[t for t in idf_tokenize(domain_carrier_text[sid]) if t not in generic_terms] for sid in slot_ids}

def neighbor_ids_for(base_id, sim_min=NEIGHBOR_SIM_MIN, top_k=NEIGHBOR_TOP_K):
    i=slot_ids.index(base_id)
    pairs=[]
    for j,sid in enumerate(slot_ids):
        if sid==base_id:
            continue
        sim=float(slot_sim[i,j])
        if sim >= sim_min:
            pairs.append((sim,sid))
    pairs=sorted(pairs, reverse=True)
    return [sid for sim,sid in pairs[:top_k]]

domain_neighbors={sid:neighbor_ids_for(sid) for sid in slot_ids}

domain_carriers_df=pd.DataFrame([
    {"base_id":sid,"neighbors":" | ".join(domain_neighbors[sid]),"carrier_terms":" ".join(carrier_terms_by_sid[sid][:30]),"carrier_preview":domain_carrier_text[sid][:220].replace("\n"," / ")}
    for sid in slot_ids
])
display(domain_carriers_df)
print("Generic discounted terms:", sorted(list(generic_terms))[:100])


In [ ]:
# Cache per-row / per-slot domain fit tensors once.
# domain_cache[row_index][slot_index, choice_index]

domain_cache=[]
v60_vector_cache=[]

for pack in tqdm(precomp, desc="v63 cache domain tensors"):
    row=pack["row"]
    choices=row["choices"]
    choice_vecs=encode_norm(choices)
    choice_tokens=[set(idf_tokenize(c)) for c in choices]

    S=len(slot_ids)
    C=len(choices)
    domain_tensor=np.zeros((S,C), dtype=np.float32)

    for si,sid in enumerate(slot_ids):
        emb_scores=(choice_vecs @ carrier_vec_by_sid[sid]).astype(np.float32)
        ov_scores=idf_overlap_scores_cached(choice_tokens, carrier_terms_by_sid[sid])
        domain_tensor[si]=(0.60*zscore_local(emb_scores)+0.40*zscore_local(ov_scores)).astype(np.float32)

    v60_vector_cache.append(compute_vectors(pack, alpha=0.0))
    domain_cache.append(domain_tensor)

print("cached rows:", len(domain_cache))


In [ ]:
def v63_vectors_from_cache(pack_idx, lambda_b=1.0, lambda_p=0.50, lambda_d=1.0, lambda_n=0.75):
    pack=precomp[pack_idx]
    row=pack["row"]
    base_id=row["base_id"]
    real_si=slot_ids.index(base_id)

    v=v60_vector_cache[pack_idx]
    domain_tensor=domain_cache[pack_idx]

    raw_scores=v["raw_scores"]
    domain_pos=domain_tensor[real_si]

    neigh=domain_neighbors[base_id]
    if neigh:
        neigh_indices=[slot_ids.index(n) for n in neigh]
        neighbor_matrix=domain_tensor[neigh_indices]
        domain_neg=neighbor_matrix.max(axis=0)
        neighbor_argmax=[neigh[int(np.argmax(neighbor_matrix[:,i]))] for i in range(domain_tensor.shape[1])]
    else:
        domain_neg=np.zeros(domain_tensor.shape[1], dtype=np.float32)
        neighbor_argmax=[""]*domain_tensor.shape[1]

    # prompt/slot anchor is the positive domain carrier channel here
    prompt_slot_scores=domain_pos

    compiler_scores=(
        lambda_b*zscore_local(raw_scores)
        + lambda_p*zscore_local(prompt_slot_scores)
        + lambda_d*zscore_local(domain_pos)
        - lambda_n*zscore_local(domain_neg)
    ).astype(np.float32)

    compiler_pred, compiler_margin=argmax_margin(compiler_scores)
    domain_pos_pred, domain_pos_margin=argmax_margin(domain_pos)
    domain_neg_pred, domain_neg_margin=argmax_margin(domain_neg)

    real=pack["branch_tensor"][real_si]
    compiler_support=support_for_pred(real, compiler_pred)
    compiler_support += int(domain_pos_pred==compiler_pred)
    compiler_support += int(v["ev_cons_pred"]==compiler_pred)

    return {
        **v,
        "domain_pos":domain_pos,
        "domain_neg":domain_neg,
        "neighbor_argmax":neighbor_argmax,
        "prompt_slot_scores":prompt_slot_scores,
        "compiler_scores":compiler_scores,
        "compiler_pred":compiler_pred,
        "compiler_margin":compiler_margin,
        "compiler_support":compiler_support,
        "domain_pos_pred":domain_pos_pred,
        "domain_pos_margin":domain_pos_margin,
        "domain_neg_pred":domain_neg_pred,
        "domain_neg_margin":domain_neg_margin,
    }

def decide_v63(v, support_min, margin_min, allow_raw_fallback=True):
    base=v["base_pred"]
    raw=v["raw_pred"]
    comp=v["compiler_pred"]

    raw_safe=(v["raw_support"]>=1 and v["raw_margin"]>=0.0)
    comp_safe=(v["compiler_support"]>=support_min and v["compiler_margin"]>=margin_min)

    if comp_safe:
        return comp,"domain_compiler_safe",0
    if allow_raw_fallback and raw_safe:
        return raw,"raw_fallback_from_compiler",0
    return base,"omega_no_safe_domain_path_keep_base",1


In [ ]:
# Fast v63 sweep. No model calls and no embeddings inside this loop.
v63_records=[]

for lambda_b in tqdm(LAMBDA_B_SWEEP, desc="v63 lambda_b"):
    for lambda_p in LAMBDA_P_SWEEP:
        for lambda_d in LAMBDA_D_SWEEP:
            for lambda_n in LAMBDA_N_SWEEP:
                vectors=[v63_vectors_from_cache(i, lambda_b=lambda_b, lambda_p=lambda_p, lambda_d=lambda_d, lambda_n=lambda_n) for i in range(len(precomp))]

                for support_min in DOMAIN_SUPPORT_SWEEP:
                    for margin_min in DOMAIN_MARGIN_SWEEP:
                        tmp_rows=[]
                        for pack,v in zip(precomp,vectors):
                            row=pack["row"]
                            pred,reason,omega=decide_v63(v, support_min, margin_min, allow_raw_fallback=True)
                            base_pred=v["base_pred"]
                            tmp_rows.append({
                                "id":row["id"],
                                "base_id":row["base_id"],
                                "band":row["band"],
                                "gold_idx":row["answer_idx"],
                                "base_correct":int(base_pred==row["answer_idx"]),
                                "raw_correct":int(v["raw_pred"]==row["answer_idx"]),
                                "domain_pos_correct":int(v["domain_pos_pred"]==row["answer_idx"]),
                                "compiler_correct":int(v["compiler_pred"]==row["answer_idx"]),
                                "ev_cons_correct":int(v["ev_cons_pred"]==row["answer_idx"]),
                                "krrb_correct":int(pred==row["answer_idx"]),
                                "omega":omega,
                                "helped":int(base_pred!=row["answer_idx"] and pred==row["answer_idx"]),
                                "hurt":int(base_pred==row["answer_idx"] and pred!=row["answer_idx"]),
                            })

                        tmp=pd.DataFrame(tmp_rows)
                        v63_records.append({
                            "lambda_b":lambda_b,
                            "lambda_p":lambda_p,
                            "lambda_d":lambda_d,
                            "lambda_n":lambda_n,
                            "support_min":support_min,
                            "margin_min":margin_min,
                            "n":len(tmp),
                            "base_acc":tmp.base_correct.mean(),
                            "raw_acc":tmp.raw_correct.mean(),
                            "domain_pos_acc":tmp.domain_pos_correct.mean(),
                            "compiler_acc":tmp.compiler_correct.mean(),
                            "ev_cons_acc":tmp.ev_cons_correct.mean(),
                            "krrb_acc":tmp.krrb_correct.mean(),
                            "gain_vs_base":tmp.krrb_correct.mean()-tmp.base_correct.mean(),
                            "helped":int(tmp.helped.sum()),
                            "hurt":int(tmp.hurt.sum()),
                            "omega_count":int(tmp.omega.sum()),
                        })

v63_sweep_df=pd.DataFrame(v63_records)
v63_ranked=v63_sweep_df.sort_values(["hurt","krrb_acc","helped","omega_count"], ascending=[True,False,False,True])
display(v63_ranked.head(60))
v63_zero_hurt=v63_ranked[v63_ranked.hurt==0]
v63_best_config=(v63_zero_hurt.iloc[0] if len(v63_zero_hurt) else v63_ranked.iloc[0]).to_dict()
v63_best_config


In [ ]:
def v63_final_run(config):
    lambda_b=float(config["lambda_b"])
    lambda_p=float(config["lambda_p"])
    lambda_d=float(config["lambda_d"])
    lambda_n=float(config["lambda_n"])
    support_min=int(config["support_min"])
    margin_min=float(config["margin_min"])

    out_rows=[]
    tables={}

    for i,pack in enumerate(precomp):
        row=pack["row"]
        v=v63_vectors_from_cache(i, lambda_b=lambda_b, lambda_p=lambda_p, lambda_d=lambda_d, lambda_n=lambda_n)
        pred,reason,omega=decide_v63(v, support_min, margin_min, allow_raw_fallback=True)

        real_slot=slot_by_base_id[row["base_id"]]
        leaks=exact_phrase_leak(real_slot,row["choices"])
        gold_leak=leaks[row["answer_idx"]]
        gold_overlap=jaccard(row["choices"][row["answer_idx"]], real_slot.as_text())
        max_choice_overlap=max(jaccard(c, real_slot.as_text()) for c in row["choices"])

        tab=table_for_row(row, pack["branch_tensor"], pack["base_scores"], pack["text_scores"], alpha=0.0)
        tab["domain_pos_score"]=v["domain_pos"]
        tab["domain_neg_score"]=v["domain_neg"]
        tab["domain_neighbor_argmax"]=v["neighbor_argmax"]
        tab["compiler_score"]=v["compiler_scores"]
        tables[row["id"]]=tab

        def pick(prefix,p):
            return {f"{prefix}_idx":p, f"{prefix}_choice":row["choices"][p], f"{prefix}_source_id":row["choice_source_ids"][p], f"{prefix}_correct":int(p==row["answer_idx"])}

        rec={
            "id":row["id"],
            "base_id":row["base_id"],
            "band":row["band"],
            "gold_idx":row["answer_idx"],
            "gold_choice":row["choices"][row["answer_idx"]],
            "gold_source_id":row["choice_source_ids"][row["answer_idx"]],
            "choice_source_ids":"|".join(row["choice_source_ids"]),
            "lambda_b":lambda_b,
            "lambda_p":lambda_p,
            "lambda_d":lambda_d,
            "lambda_n":lambda_n,
            "support_min":support_min,
            "margin_min":margin_min,
            "reason":reason,
            "omega":omega,
            "base_margin":v["base_margin"],
            "raw_margin":v["raw_margin"],
            "raw_support":v["raw_support"],
            "domain_pos_margin":v["domain_pos_margin"],
            "domain_neg_margin":v["domain_neg_margin"],
            "compiler_margin":v["compiler_margin"],
            "compiler_support":v["compiler_support"],
            "neighbors":"|".join(domain_neighbors[row["base_id"]]),
            "gold_exact_phrase_leak":gold_leak,
            "gold_slot_overlap":gold_overlap,
            "max_choice_slot_overlap":max_choice_overlap,
            "need_slot_text":real_slot.as_text(),
            "domain_carrier_text":domain_carrier_text[row["base_id"]],
        }

        rec.update(pick("base",v["base_pred"]))
        rec.update(pick("raw",v["raw_pred"]))
        rec.update(pick("domain_pos",v["domain_pos_pred"]))
        rec.update(pick("compiler",v["compiler_pred"]))
        rec.update(pick("ev_cons",v["ev_cons_pred"]))
        rec.update(pick("krrb",pred))
        out_rows.append(rec)

    df=pd.DataFrame(out_rows)
    for mode in ["raw","domain_pos","compiler","ev_cons","krrb"]:
        df[f"{mode}_helped"]=((df.base_correct==0)&(df[f"{mode}_correct"]==1)).astype(int)
        df[f"{mode}_hurt"]=((df.base_correct==1)&(df[f"{mode}_correct"]==0)).astype(int)

    return df,tables

v63_results_df, v63_branch_tables = v63_final_run(v63_best_config)

v63_summary=pd.DataFrame([{
    "n":len(v63_results_df),
    "n_base_items":v63_results_df.base_id.nunique(),
    **{k:v63_best_config[k] for k in ["lambda_b","lambda_p","lambda_d","lambda_n","support_min","margin_min"]},
    "base_acc":v63_results_df.base_correct.mean(),
    "raw_acc":v63_results_df.raw_correct.mean(),
    "domain_pos_acc":v63_results_df.domain_pos_correct.mean(),
    "compiler_acc":v63_results_df.compiler_correct.mean(),
    "ev_cons_acc":v63_results_df.ev_cons_correct.mean(),
    "krrb_acc":v63_results_df.krrb_correct.mean(),
    "krrb_gain_vs_base":v63_results_df.krrb_correct.mean()-v63_results_df.base_correct.mean(),
    "krrb_helped":int(v63_results_df.krrb_helped.sum()),
    "krrb_hurt":int(v63_results_df.krrb_hurt.sum()),
    "omega_count":int(v63_results_df.omega.sum()),
    "gold_exact_phrase_leaks":int(v63_results_df.gold_exact_phrase_leak.sum()),
}])

v63_by_band=v63_results_df.groupby("band")[["base_correct","raw_correct","domain_pos_correct","compiler_correct","ev_cons_correct","krrb_correct","krrb_helped","krrb_hurt","omega"]].mean().reset_index()
v63_by_base_item=v63_results_df.groupby(["base_id","band","gold_source_id"])[["base_correct","raw_correct","domain_pos_correct","compiler_correct","ev_cons_correct","krrb_correct","krrb_helped","krrb_hurt","omega"]].mean().reset_index()
v63_interesting=v63_results_df[(v63_results_df.raw_correct==0)|(v63_results_df.compiler_correct==0)|(v63_results_df.krrb_correct==0)|(v63_results_df.krrb_idx!=v63_results_df.base_idx)|(v63_results_df.omega==1)].copy()

display(v63_summary)
display(v63_by_band)
display(v63_by_base_item.sort_values("krrb_correct").head(24))
display(v63_interesting[["id","base_id","band","gold_choice","choice_source_ids","base_choice","base_source_id","base_correct","base_margin","raw_choice","raw_source_id","raw_correct","raw_margin","raw_support","domain_pos_choice","domain_pos_source_id","domain_pos_correct","domain_pos_margin","compiler_choice","compiler_source_id","compiler_correct","compiler_margin","compiler_support","ev_cons_choice","ev_cons_source_id","ev_cons_correct","krrb_choice","krrb_source_id","krrb_correct","reason","omega","neighbors"]].head(160))


In [ ]:
v63_out=Path(V63_OUTPUT_DIR)
v63_results_df.to_csv(v63_out/"results.csv",index=False)
v63_summary.to_csv(v63_out/"summary.csv",index=False)
v63_sweep_df.to_csv(v63_out/"domain_carrier_sweep.csv",index=False)
v63_ranked.to_csv(v63_out/"domain_carrier_sweep_ranked.csv",index=False)
v63_by_band.to_csv(v63_out/"by_band.csv",index=False)
v63_by_base_item.to_csv(v63_out/"by_base_item.csv",index=False)
v63_interesting.to_csv(v63_out/"interesting_cases.csv",index=False)
domain_carriers_df.to_csv(v63_out/"domain_carriers.csv",index=False)

for row_id,tab in v63_branch_tables.items():
    safe=row_id.replace("/","_")
    tab.to_csv(v63_out/f"branches_{safe}.csv",index=False)

print("Saved v63 outputs in", v63_out)
print("results.csv, summary.csv, domain_carrier_sweep.csv, domain_carrier_sweep_ranked.csv, by_band.csv, by_base_item.csv, interesting_cases.csv, domain_carriers.csv, branches_*.csv")


In [ ]:
v63_summary[["base_acc","raw_acc","domain_pos_acc","compiler_acc","ev_cons_acc","krrb_acc"]].T.plot(kind="bar", legend=False)
plt.title("v63 Cached Domain-Carrier Compiler Accuracy")
plt.ylabel("accuracy")
plt.xticks(rotation=30,ha="right")
plt.tight_layout()
plt.show()

v63_by_band.set_index("band")[["base_correct","raw_correct","compiler_correct","krrb_correct"]].plot(kind="bar", figsize=(14,5))
plt.title("v63 Accuracy by Band")
plt.ylabel("accuracy")
plt.xticks(rotation=30,ha="right")
plt.tight_layout()
plt.show()

pd.DataFrame({
    "raw":[v63_results_df.raw_helped.sum(),v63_results_df.raw_hurt.sum()],
    "compiler":[v63_results_df.compiler_helped.sum(),v63_results_df.compiler_hurt.sum()],
    "ev_cons":[v63_results_df.ev_cons_helped.sum(),v63_results_df.ev_cons_hurt.sum()],
    "krrb":[v63_results_df.krrb_helped.sum(),v63_results_df.krrb_hurt.sum()],
}, index=["helped","hurt"]).plot(kind="bar")
plt.title("v63 Help vs Hurt")
plt.ylabel("count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(v63_ranked[v63_ranked.hurt==0].head(50)[["lambda_b","lambda_p","lambda_d","lambda_n","support_min","margin_min","krrb_acc","gain_vs_base","helped","hurt","omega_count","raw_acc","domain_pos_acc","compiler_acc","ev_cons_acc"]])

tree_cases=v63_results_df[v63_results_df.base_id=="adv_tree_01"]["id"].tolist()
for row_id in tree_cases:
    print("="*100)
    print("TREE CASE:", row_id)
    display(v63_results_df[v63_results_df.id==row_id][["id","choice_source_ids","gold_choice","raw_choice","raw_correct","domain_pos_choice","domain_pos_source_id","domain_pos_correct","compiler_choice","compiler_source_id","compiler_correct","compiler_margin","compiler_support","krrb_choice","krrb_source_id","krrb_correct","reason","neighbors"]])
    display(v63_branch_tables[row_id].sort_values("compiler_score", ascending=False))


## v63 Readout

If v63 reaches:

$$
\text{krrb\_acc}=1.0,\qquad \text{krrb\_hurt}=0
$$

then the missing structure was a cached domain-carrier compiler.

If v63 remains at:

$$
0.958333
$$

then the diagnostic is stronger:

$$
\Omega_{\text{tree/moore}}
\neq
\text{score-weight problem}
$$

It means the slot must be produced by a model before choice scoring.

Next architecture:

$$
\text{prompt}
\rightarrow
\left(
\text{domain carrier},
\text{forbidden carriers},
\text{abstract slot},
\text{failure modes}
\right)
\rightarrow
\text{candidate scoring}
$$

That is the practical Nexus AI branch:

$$
\boxed{
\text{compile the missing-shape contract first; answer second}
}
$$
